In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from rag_helper import RAGBAse
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBAse):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


In [4]:
vector_assistant = RAGVector(
    index=vs_index,
    embedder=model,
    llm_client=openai_client
)

In [5]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. You can start learning and submitting homework as long as the submission form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [7]:
vs_index.close()